In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Dropout, Input

In [ ]:
# 1. Load Google stock (CSV is already chronological)
DATA_PATH = "Datasets/google.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["Date"], index_col="Date")[["Close"]]
print(df.head())
print(f"Rows: {len(df)}")

plt.figure(figsize=(16, 6))
plt.title("Google Stock Close Price History")
plt.plot(df["Close"])
plt.xlabel("Date")
plt.ylabel("Close Price USD ($)")
plt.show()

In [ ]:
# 2. Preprocessing — predict next Close from past LOOKBACK closes
LOOKBACK = 5  # lower if you have very few rows
TRAIN_FRAC = 0.8

data = df.values.astype(np.float64)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

training_data_len = int(np.ceil(len(data) * TRAIN_FRAC))
print(f"Training data length: {training_data_len}")

train_data = scaled_data[0:training_data_len, :]
if len(train_data) <= LOOKBACK:
    raise ValueError(
        f"Training rows ({len(train_data)}) must exceed LOOKBACK ({LOOKBACK}). "
        "Use more data or lower LOOKBACK."
    )

x_train, y_train = [], []
for i in range(LOOKBACK, len(train_data)):
    x_train.append(train_data[i - LOOKBACK : i, 0])
    y_train.append(train_data[i, 0])

x_train = np.asarray(x_train, dtype=np.float32).reshape(-1, LOOKBACK, 1)
y_train = np.asarray(y_train, dtype=np.float32).ravel()  # 1-D targets (Keras 3 / optree)
print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")

In [ ]:
# 3. RNN model (SimpleRNN stack from reference)
model = Sequential(
    [
        Input(shape=(LOOKBACK, 1)),
        SimpleRNN(50, return_sequences=True),
        Dropout(0.2),
        SimpleRNN(50, return_sequences=False),
        Dropout(0.2),
        Dense(25, activation="relu"),
        Dense(1),
    ]
)
model.summary()
model.compile(optimizer="adam", loss="mean_squared_error")

In [ ]:
# 4. Train
history = model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=10,
    verbose=1,
)

In [ ]:
# 5. Test windows (include last LOOKBACK rows before test region)
test_data = scaled_data[training_data_len - LOOKBACK :, :]
y_test = data[training_data_len:, :]  # actual unscaled closes in test period

x_test = []
for i in range(LOOKBACK, len(test_data)):
    x_test.append(test_data[i - LOOKBACK : i, 0])
x_test = np.asarray(x_test, dtype=np.float32).reshape(-1, LOOKBACK, 1)

predictions = model.predict(x_test, verbose=0)
predictions = scaler.inverse_transform(predictions)

rmse = np.sqrt(np.mean((predictions - y_test) ** 2))
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

In [ ]:
# 6. Plot train vs actual vs predicted
train = df.iloc[:training_data_len]
valid = df.iloc[training_data_len:].copy()
valid["Predictions"] = predictions

plt.figure(figsize=(16, 6))
plt.title("Google Stock Price Prediction — RNN")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Close Price USD ($)", fontsize=12)
plt.plot(train["Close"], label="Train")
plt.plot(valid["Close"], label="Validation (actual)")
plt.plot(valid["Predictions"], label="Predictions")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()